# KuwaPIPE — PS 侧控制脚本（PYNQ）

本 Notebook 负责四件事：

1. 加载 Overlay（`base.bit` + `base.hwh`）；
2. 通过 I2C 初始化 OV5640，输出 **800x480 RGB565**；
3. 配置 `VideoMode(640, 480, 24)`，建立「摄像头 -> PL 处理链 -> HDMI」直通；
4. 抓帧保存 + 对硬件描述做自检（确认跑的是自研模块）。

硬件：xc7z020clg400-2（PYNQ-Z2 兼容板）+ OV5640 + HDMI 显示器。

> **显示分辨率只能用 640x480**：PYNQ 的 DVI 输出只有
> `(640,480) (800,600) (1280,720) (1280,1024) (1920,1080)` 五种模式，
> 上游链路的 800x480 不在其中，写 `VideoMode(800, 480, 24)` 会在 `hdmi.configure()`
> 抛 `ValueError: Invalid Output resolution 800x480`。
> 因此每行只显示前 640 像素，**右侧 20% 被裁掉** —— 这是显示通路的硬限制，不是代码问题。

> **运行前**：把 `board/bitstream/base.bit` 与 `base.hwh` 拷到本 notebook 同一目录（`BIT_PATH` 默认 `./base.bit`，PYNQ 按同名找 `.hwh`）。
> 本文件里保存了**真实上板输出**（抓帧图 + 耗时日志）；不想要输出时可用 `jupyter nbconvert --ClearOutputPreprocessor.enabled=True --to notebook --inplace` 清掉。


In [ ]:
import os
from datetime import datetime

import numpy as np
import PIL.Image

from pynq.overlays.base import BaseOverlay
from pynq.lib.iic import AxiIIC
from pynq.lib.ov5640 import OV5640
from pynq.lib.video import PIXEL_RGB, VideoMode

# ---------------- 配置 ----------------
BIT_PATH    = "./base.bit"
HWH_PATH    = "./base.hwh"
FRAME_W     = 640          # 显示模式受 PYNQ DVI 限制（800x480 不支持），右侧会裁 20%
FRAME_H     = 480
FRAME_BPP   = 24
CAPTURE_DIR = "./captures"
os.makedirs(CAPTURE_DIR, exist_ok=True)

# ---------------- 加载 Overlay ----------------
base = BaseOverlay(BIT_PATH)
cam  = base.video.camera_in
hdmi = base.video.hdmi_out
print("overlay loaded:", BIT_PATH)

In [ ]:
# ---------------- OV5640 初始化（I2C）----------------
iic = AxiIIC(base.ip_dict["cam_iic"])
sensor = OV5640(0x3C, iic)     # OV5640 7-bit I2C 地址
sensor.init()                  # 寄存器表固定输出 800x480 RGB565
print("OV5640 initialised (800x480)")

In [ ]:
# ---------------- 视频通路 ----------------
mode = VideoMode(FRAME_W, FRAME_H, FRAME_BPP)
cam.configure(mode, PIXEL_RGB)
hdmi.configure(cam.mode, PIXEL_RGB)
cam.start()
hdmi.start()
cam.tie(hdmi)                  # 直通显示；PL 链路内已含 中值 -> Kuwahara -> 风格化
print("video running:", cam.mode)

## 抓一帧保存

`camera_in.readframe()` 返回的是 PL 送进 VDMA 的帧；直通模式下它同时出现在 HDMI 上。

In [ ]:
# ---------------- 抓一帧并保存 ----------------
frame = cam.readframe()        # numpy (H, W, 3), uint8
img = PIL.Image.fromarray(frame)
out = os.path.join(CAPTURE_DIR,
                   "frame_%s.png" % datetime.now().strftime("%Y%m%d_%H%M%S"))
img.save(out)
print("saved:", out, frame.shape, frame.dtype)
img

In [ ]:
# ---------------- 硬件自检：确认跑的是自研模块 ----------------
with open(HWH_PATH, "r", encoding="utf-8", errors="ignore") as f:
    hwh = f.read()

checks = {
    "ov5640_capture (自研采集 IP)":  "ov5640_capture" in hwh,
    "video_kuwahara (Kuwahara)":     "video_kuwahara" in hwh,
    "video_mode_switch (按键切换)":   "video_mode_switch" in hwh,
    "video_style_bypass (风格化旁路)": "video_style_bypass" in hwh,
}
for k, v in checks.items():
    print("%-34s %s" % (k, "OK" if v else "MISSING"))

In [ ]:
# ---------------- 按键状态（切换由 PL 完成，这里只读）----------------
try:
    raw = base.buttons.read()
    print("buttons raw =", bin(raw))
    print("提示：PL_KEY0 由 PL 侧消抖并切换 处理/直通，LED2 亮 = 直通")
except Exception as e:
    print("buttons unavailable:", e)

In [ ]:
# ===== PS 端 Kuwahara 保边平滑（对应 style_cartoon_640x480.py 的 _kuwahara）=====
# 完整卡通链路：相机 -> 中值(FPGA采集端) -> DDR -> [本cell:Kuwahara] -> DDR
#             -> gamma+饱和度+量化(FPGA显示端) -> HDMI
import numpy as np
import time

# 兼容 notebook：notebook 里变量叫 cam/hdmi，这里统一成本 cell 用的名字
camera_in = base.video.camera_in
hdmi_out  = base.video.hdmi_out

def luma_q8(rgb):
    """亮度 Q8：77*R + 150*G + 29*B"""
    r = rgb[..., 0].astype(np.int64)
    g = rgb[..., 1].astype(np.int64)
    b = rgb[..., 2].astype(np.int64)
    return 77 * r + 150 * g + 29 * b

def kuwahara(rgb, r=3):
    """
    Kuwahara 保边平滑：每个像素 4 个 (r+1)x(r+1) 象限，选亮度方差最小的象限，
    取该象限 RGB 均值输出。积分图实现，O(1)/象限。r=3 -> 4x4，/16 精确。
    """
    a = rgb.astype(np.float64)
    h, w = rgb.shape[:2]
    y = luma_q8(rgb).astype(np.float64) / 256.0

    def integ(x):
        ii = np.zeros((x.shape[0] + 1, x.shape[1] + 1), np.float64)
        ii[1:, 1:] = x.cumsum(0).cumsum(1)
        return ii

    Yp = np.pad(y, r, mode="edge")
    Ip = np.pad(a, ((r, r), (r, r), (0, 0)), mode="edge")
    iy, iy2 = integ(Yp), integ(Yp * Yp)
    icol = [integ(Ip[..., c]) for c in range(3)]

    Yg = np.arange(h)[:, None]
    Xg = np.arange(w)[None, :]
    n = (r + 1) * (r + 1)

    best = np.full((h, w), np.inf)
    out = np.zeros((h, w, 3))
    for dy, dx in [(0, 0), (0, r), (r, 0), (r, r)]:
        y0, x0 = Yg + dy, Xg + dx
        s  = iy [y0 + r + 1, x0 + r + 1] - iy [y0, x0 + r + 1] - iy [y0 + r + 1, x0] + iy [y0, x0]
        s2 = iy2[y0 + r + 1, x0 + r + 1] - iy2[y0, x0 + r + 1] - iy2[y0 + r + 1, x0] + iy2[y0, x0]
        m = s / n
        var = np.maximum(s2 / n - m * m, 0.0)
        sel = var < best
        best = np.where(sel, var, best)
        for c in range(3):
            ii = icol[c]
            sc = ii[y0 + r + 1, x0 + r + 1] - ii[y0, x0 + r + 1] - ii[y0 + r + 1, x0] + ii[y0, x0]
            out[..., c] = np.where(sel, sc / n, out[..., c])
    return np.clip(np.round(out), 0, 255).astype(np.uint8)

# ---- 循环：读一帧 -> Kuwahara -> 写回 HDMI（30 帧后自动停止）----
N_FRAMES = 30
t0 = time.perf_counter()
for n in range(1, N_FRAMES + 1):
    frame = camera_in.readframe()          # Frame 对象（带 flush() 的 numpy 子类）
    t1 = time.perf_counter()
    frame[:] = kuwahara(frame, r=3)        # 原地写回，保持 Frame 类型
    hdmi_out.writeframe(frame)             # frame 还是 Frame，不报错
    dt = (time.perf_counter() - t1) * 1e3
    if n % 5 == 0 or n == N_FRAMES:
        print("  %d/%d  (kuwahara %.1f ms/frame)" % (n, N_FRAMES, dt))
print("done: %d frames in %.1f s" % (N_FRAMES, time.perf_counter() - t0))

## 收尾

停止前建议先执行下面的单元，避免 DMA 占用。

In [ ]:
# ---------------- 收尾 ----------------
hdmi.stop()
cam.stop()
print("video stopped")